# 5.4. Stage 5: Monthly Aggregation — Full Dataset

**Stage:** 5.4 of 5 (final)

**Purpose:** Aggregate the daily full pickles produced by Stage 5.3 into monthly Parquet files. Produces the final research dataset used in the paper.

**Input:** `data/stage_05/intermediate/pkl_daily_full/ua-YYYY-MM-DD.pkl`

**Output:**
- `data/stage_05/processed/parquet_monthly_full/YYYY-M.parquet` — monthly Parquet (all languages)
- `data/stage_05/processed/json_monthly_ua_full/YYYY-M.parquet` — monthly Parquet (Ukrainian only)

**Run:** Set `start_date` and `end_date` to cover the months you want to generate, then execute all cells top to bottom.

In [1]:
import datetime
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append("../code")
import general as g
g.clean_memory()

Enable autoreload, add `code/` to the Python path, and trigger memory cleanup.

In [5]:
import stage1 as st1
import stage2 as st2
import stage3 as st3
import pandas as pd

Import stage modules and pandas.

In [3]:
g.check_folder_exists(g.Config.STAGE5_MONTHLY_FULL_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_FULL_JSON_UA_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_FULL_PARQUET_PATH)

Ensure all monthly full output folders exist before writing.

In [6]:
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)
process_df = process_df.sort_values(by='input_path').reset_index(drop=True)
process_df.tail(50)

,input_path,region_path,rejoin_path,rejoin_status
0,ua-2024-01-01,../data/stage_01/intermediate/id_region_click\...,../data/stage_05/intermediate/pkl_daily_full\u...,complete


Load the Stage 5 full process tracker, sorted chronologically.

In [9]:
import pandas as pd

start_date = "2024-01-01"
end_date = "2024-02-01"

date_range = pd.date_range(start=start_date, end=end_date, freq='ME')
df = pd.DataFrame({'year': date_range.year, 'month': date_range.month})
df

,year,month
0,2024,1


Define the date range to process — one row per calendar month. Adjust `start_date` and `end_date` to cover the full dataset.

> **Demo:** The default range `2024-01-01` to `2024-02-01` covers the single demo file month.

In [10]:
import os

selected_columns = ['id', 'title', 'description', 'min_salary', 'max_salary', 'currency', 'salary_rate', 'date_created', 'date_expired', 'clean_title', 'clean_desc', 'title_lang', 'desc_lang', 'skill_ids', 'skill_labels','job_type', 'classified_code', 'classified_title', 'skill_labels_en', 'classified_title_clean', 'extract_type', 'esco_title', 'esco_skills', 'esco_id', 'esco_code',  'number_of_clicks', 'date', 'region_original', 'city', 'district', 'region', 'country', 'latitude', 'longitude']

for _, row in df.iterrows():

    month_data = pd.DataFrame(columns=selected_columns)
    print(f"Month: {df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.")

    parquet_path = os.path.join(g.Config.STAGE5_MONTHLY_FULL_PARQUET_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.parquet")

    if os.path.exists(parquet_path):
        print(f"Skipping month {df.loc[_ ,"year"]}-{df.loc[_ ,"month"]} - already exists.")
        continue


    started = False
    for __, prow in process_df.iterrows():

        if pd.isna(prow["rejoin_path"]):
            print(f"Skipping row with missing path: {prow.name}")
            continue

        # Verify path exists
        if not os.path.exists(prow["rejoin_path"]):
            print(f"File not found: {prow['rejoin_path']}")
            continue

        data = pd.read_pickle(prow["rejoin_path"])
        data['id'] = data['id'].astype(str)
        data["date"] = pd.to_datetime(data["date"])

        if len(data) == 0:
            continue

        first_date = data.iloc[0]["date"]
        y = first_date.year
        m = first_date.month

        if y == row["year"] and m == row["month"]:
            print(f"{first_date}: working...")
            if month_data.shape[0] == 0:
                started = True
                month_data = data[selected_columns]
                continue
            else:
                data = data[selected_columns]
                month_data = pd.concat([month_data, data], ignore_index=True)
        else:
            #print(f"skip...")
            if started:
                break

    if month_data.shape[0] > 0:
       # json_path = os.path.join(Config.STAGE5_MONTHLY_FULL_JSON_PATH, #f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.json")
       # month_data.to_json(json_path, orient="records")

        parquet_path = os.path.join(g.Config.STAGE5_MONTHLY_FULL_PARQUET_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.parquet")
        month_data.to_parquet(parquet_path)

        month_data = month_data[month_data["country"] == "Ukraine"]
        parquet_path = os.path.join(g.Config.STAGE5_MONTHLY_FULL_JSON_UA_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.parquet")
        month_data.to_parquet(parquet_path)
        #ua_json_path = os.path.join(Config.STAGE5_MONTHLY_FULL_JSON_UA_PATH, f"{df.loc[_ ,"year"]}-{df.loc[_ ,"month"]}.json")
        #month_data.to_json(ua_json_path, orient="records")

        print(f"{_} - date {df.loc[_ ,"year"]}-{df.loc[_ ,"month"]} processing complete!")


print("All done!")

Month: 2024-1.
2024-01-01 00:00:00: working...
0 - date 2024-1 processing complete!
All done!


**Main monthly aggregation loop** — same pattern as Stage 5.2 but for the full dataset. For each month, all matching daily full pickles are concatenated using `selected_columns`. Months already written to Parquet are skipped automatically.

Outputs per month:
- `parquet_monthly_full/YYYY-M.parquet` — full dataset (all languages)
- `json_monthly_ua_full/YYYY-M.parquet` — Ukrainian-only subset

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.